# SAE training reproducibility & seed variance

How much of a trained SAE is determined by the **RNG seed** vs. fixed once the seed is pinned?
This notebook analyses the runs produced by `experiments/repro_experiment.sh`, which trains the
**same** Pythia SAE config (`pythia-160m-deduped`, one member, 100M tokens) many times.

**Two questions:**

1. **Determinism (same seed).** Two independent processes with the *same* seed 0 and
   `--deterministic` (`saebench_pythia_repro_seed0` vs `..._seed0_dup`). With our end-to-end
   seeding they should be **bit-identical** — identical weights *and* identical SAEBench evals.
2. **Seed variance (different seeds).** `N` runs with seeds `0..N-1`
   (`saebench_pythia_repro_seed{0..N-1}`). We measure the spread, across seeds, of
   - every SAEBench eval metric (core + sparse_probing), and
   - pairwise **weight similarity** (permutation-invariant, since feature order differs per seed).

Background: SAELens' config `seed` is a no-op in the training path; the two real RNG draws
(SAE weight init `kaiming_uniform_`, and the activation mixing-buffer `torch.randperm`) both
use the global torch RNG, which our `train_saebench_replication.py --seed` now pins.

In [ ]:
from pathlib import Path
import glob, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from safetensors import safe_open

# ── Config: must match experiments/repro_experiment.sh ────────────────────────
RESULTS = Path("../experiments/results").resolve()
MODEL   = "pythia"
WIDTH, K = 16384, 40
MEMBER  = f"w{WIDTH}_k{K}"
NSEEDS  = 10                       # variance set: seeds 0..NSEEDS-1
DUP_TAG = "repro_seed0_dup"        # determinism partner of seed 0
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"

def run_dir(tag: str) -> Path:
    return RESULTS / f"saebench_{MODEL}_{tag}"

# Discover which runs actually finished (final SAE present).
seed_runs = {s: run_dir(f"repro_seed{s}") for s in range(NSEEDS)}
seed_runs = {s: d for s, d in seed_runs.items() if (d / MEMBER / "cfg.json").exists()}
dup_dir   = run_dir(DUP_TAG)
have_dup  = (dup_dir / MEMBER / "cfg.json").exists()

print(f"member          {MEMBER}")
print(f"seeds present   {sorted(seed_runs)}  ({len(seed_runs)}/{NSEEDS})")
print(f"seed0 dup       {'yes' if have_dup else 'MISSING'}  ({dup_dir.name})")
print(f"device          {DEVICE}")
if not seed_runs:
    print("\n[!] No finished runs yet — launch experiments/repro_experiment.sh first.")

## Loaders — SAE weights and SAEBench eval metrics

In [ ]:
def load_weights(d: Path, device: str = "cpu") -> dict[str, torch.Tensor]:
    """Load the final SAE's tensors (W_enc, W_dec, b_enc, b_dec, threshold) as float32."""
    p = d / MEMBER / "sae_weights.safetensors"
    out = {}
    with safe_open(str(p), framework="pt", device=device) as f:
        for k in f.keys():
            out[k] = f.get_tensor(k).float()
    return out

# Which scalar metrics to track from each eval JSON (flattened dotted paths).
CORE_KEYS = {
    "reconstruction_quality.explained_variance": "explained_var",
    "reconstruction_quality.mse":                "mse",
    "reconstruction_quality.cossim":             "cossim",
    "model_performance_preservation.ce_loss_score": "ce_loss_score",
    "sparsity.l0":                               "l0",
    "sparsity.l1":                               "l1",
    "shrinkage.l2_ratio":                        "l2_ratio",
    "misc_metrics.frac_alive":                   "frac_alive",
    "misc_metrics.freq_over_1_percent":          "freq_over_1pct",
}
SP_KEYS = {
    "sae.sae_test_accuracy":      "sp_acc",
    "sae.sae_top_1_test_accuracy":"sp_top1",
    "sae.sae_top_2_test_accuracy":"sp_top2",
    "sae.sae_top_5_test_accuracy":"sp_top5",
}

def _dig(d: dict, dotted: str):
    cur = d
    for part in dotted.split("."):
        if not isinstance(cur, dict) or part not in cur:
            return np.nan
        cur = cur[part]
    return cur

def _latest_eval_json(d: Path, eval_type: str) -> Path | None:
    files = sorted(glob.glob(str(d / "saebench_eval" / MEMBER / "eval_results" / eval_type / "*.json")))
    return Path(files[-1]) if files else None

def load_metrics(d: Path) -> dict[str, float]:
    """Flatten the tracked core + sparse_probing scalars for one run (NaN if absent)."""
    row: dict[str, float] = {}
    cj = _latest_eval_json(d, "core")
    if cj:
        m = json.loads(cj.read_text())["eval_result_metrics"]
        for path, name in CORE_KEYS.items():
            row[name] = float(_dig(m, path))
    sj = _latest_eval_json(d, "sparse_probing")
    if sj:
        m = json.loads(sj.read_text())["eval_result_metrics"]
        for path, name in SP_KEYS.items():
            row[name] = float(_dig(m, path))
    return row

print("loaders ready.")

## Part A — Determinism (same seed, two processes)

Compare `seed0` vs `seed0_dup`. With correct seeding + `--deterministic`, every tensor should
match to the bit (`max |Δ| == 0`) and every eval metric should be identical. Any nonzero
difference points to a remaining nondeterminism source (e.g. a CUDA op without a deterministic
kernel, or TF32 left on).

In [ ]:
if 0 in seed_runs and have_dup:
    wa = load_weights(seed_runs[0])
    wb = load_weights(dup_dir)
    rows = []
    for k in wa:
        diff = (wa[k] - wb[k]).abs()
        denom = wa[k].abs().max().item() or 1.0
        rows.append({
            "tensor": k,
            "shape": tuple(wa[k].shape),
            "max_abs_diff": diff.max().item(),
            "mean_abs_diff": diff.mean().item(),
            "rel_max_diff": diff.max().item() / denom,
            "exactly_equal": bool(torch.equal(wa[k], wb[k])),
        })
    wdiff = pd.DataFrame(rows).set_index("tensor")
    display(wdiff)
    all_bit_identical = bool(wdiff["exactly_equal"].all())
    print(("✅ WEIGHTS BIT-IDENTICAL across the same-seed runs."
           if all_bit_identical else
           "⚠️  weights differ — residual nondeterminism (see max_abs_diff above)."))

    # Eval-metric agreement for the same seed.
    ma, mb = load_metrics(seed_runs[0]), load_metrics(dup_dir)
    common = sorted(set(ma) & set(mb))
    edf = pd.DataFrame({
        "seed0": [ma[k] for k in common],
        "seed0_dup": [mb[k] for k in common],
    }, index=common)
    edf["abs_diff"] = (edf["seed0"] - edf["seed0_dup"]).abs()
    display(edf)
    print(("✅ EVALS IDENTICAL across the same-seed runs."
           if edf["abs_diff"].max() == 0 else
           f"⚠️  eval metrics differ (max abs diff {edf['abs_diff'].max():.3e})."))
else:
    print("Need both seed0 and seed0_dup runs for the determinism check.")

## Part B — Weight similarity across seeds (permutation-invariant)

Different seeds learn the *same* features in a **different order**, so a direct elementwise diff
is meaningless. Instead we use the standard dictionary-matching metric: for each decoder feature
in SAE A, take its **max cosine similarity** to any feature in SAE B, and average over features
(symmetrised). 1.0 = every feature in A has a near-identical twin in B; lower = the two
dictionaries genuinely diverge. We do this for the **decoder** (feature directions) and the
**encoder**, for all seed pairs, and contrast it with the same-seed (determinism) baseline.

In [ ]:
@torch.no_grad()
def mean_max_cosine(A: torch.Tensor, B: torch.Tensor, chunk: int = 2048) -> float:
    """Symmetric mean-of-max cosine similarity between two feature sets.

    A, B: (n_features, d) feature matrices (rows = features). Returns the mean over A of
    max_j cos(A_i, B_j), averaged with the B->A direction. Chunked to bound memory.
    """
    A = torch.nn.functional.normalize(A.to(DEVICE), dim=1)
    B = torch.nn.functional.normalize(B.to(DEVICE), dim=1)
    def one_way(X, Y):
        best = torch.empty(X.shape[0], device=X.device)
        for i in range(0, X.shape[0], chunk):
            sims = X[i:i+chunk] @ Y.T          # (chunk, n_Y)
            best[i:i+chunk] = sims.max(dim=1).values
        return best.mean().item()
    return 0.5 * (one_way(A, B) + one_way(B, A))

def dec_features(w):  # decoder rows are feature directions: (d_sae, d_in)
    return w["W_dec"]
def enc_features(w):  # encoder columns -> rows: (d_sae, d_in)
    return w["W_enc"].T.contiguous()

if len(seed_runs) >= 2:
    weights = {s: load_weights(d) for s, d in seed_runs.items()}
    seeds = sorted(weights)
    n = len(seeds)
    Dec = np.eye(n); Enc = np.eye(n)
    for ia, a in enumerate(seeds):
        for ib in range(ia + 1, n):
            b = seeds[ib]
            Dec[ia, ib] = Dec[ib, ia] = mean_max_cosine(dec_features(weights[a]), dec_features(weights[b]))
            Enc[ia, ib] = Enc[ib, ia] = mean_max_cosine(enc_features(weights[a]), enc_features(weights[b]))

    off = ~np.eye(n, dtype=bool)
    dec_pairs, enc_pairs = Dec[off], Enc[off]
    # Same-seed reference (should be ~1.0).
    same_seed_dec = (mean_max_cosine(dec_features(load_weights(seed_runs[0])),
                                     dec_features(load_weights(dup_dir)))
                     if have_dup else np.nan)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    im0 = ax[0].imshow(Dec, vmin=min(dec_pairs.min(), 0.0), vmax=1.0, cmap="viridis")
    ax[0].set(title="Decoder mean-max cosine\n(pairwise across seeds)",
              xticks=range(n), yticks=range(n), xticklabels=seeds, yticklabels=seeds,
              xlabel="seed", ylabel="seed")
    fig.colorbar(im0, ax=ax[0], fraction=0.046)
    im1 = ax[1].imshow(Enc, vmin=min(enc_pairs.min(), 0.0), vmax=1.0, cmap="viridis")
    ax[1].set(title="Encoder mean-max cosine\n(pairwise across seeds)",
              xticks=range(n), yticks=range(n), xticklabels=seeds, yticklabels=seeds, xlabel="seed")
    fig.colorbar(im1, ax=ax[1], fraction=0.046)
    ax[2].hist(dec_pairs, bins=15, alpha=0.7, label=f"decoder (μ={dec_pairs.mean():.3f})")
    ax[2].hist(enc_pairs, bins=15, alpha=0.7, label=f"encoder (μ={enc_pairs.mean():.3f})")
    if not np.isnan(same_seed_dec):
        ax[2].axvline(same_seed_dec, color="red", ls="--", label=f"same-seed dec ({same_seed_dec:.3f})")
    ax[2].set(title="Cross-seed feature-match distribution", xlabel="mean-max cosine", ylabel="# seed pairs")
    ax[2].legend()
    plt.tight_layout(); plt.show()

    print(f"Decoder cross-seed mean-max cosine: {dec_pairs.mean():.4f} ± {dec_pairs.std():.4f} "
          f"[{dec_pairs.min():.4f}, {dec_pairs.max():.4f}]")
    print(f"Encoder cross-seed mean-max cosine: {enc_pairs.mean():.4f} ± {enc_pairs.std():.4f} "
          f"[{enc_pairs.min():.4f}, {enc_pairs.max():.4f}]")
    if not np.isnan(same_seed_dec):
        print(f"Same-seed decoder reference     : {same_seed_dec:.4f} (≈1.0 if deterministic)")
else:
    print("Need >=2 finished seed runs for the cross-seed weight comparison.")

## Part C — Eval-metric variance across seeds

For every SAEBench metric, how much does it move when only the seed changes? We report
mean, std, and coefficient of variation (CV = std/|mean|) across the seed runs — the CV is the
"how many percent does this metric wobble per seed" number you'd quote when comparing two
methods (a real effect must exceed this seed noise to be meaningful).

In [ ]:
metrics = pd.DataFrame({s: load_metrics(d) for s, d in seed_runs.items()}).T
metrics.index.name = "seed"

if not metrics.empty:
    metrics = metrics.sort_index()
    display(metrics.round(5))

    summary = pd.DataFrame({
        "mean": metrics.mean(),
        "std":  metrics.std(ddof=1) if len(metrics) > 1 else metrics.std(ddof=0),
        "min":  metrics.min(),
        "max":  metrics.max(),
    })
    summary["cv_%"] = (summary["std"] / summary["mean"].abs()).replace([np.inf], np.nan) * 100
    summary["range_%"] = (summary["max"] - summary["min"]) / summary["mean"].abs() * 100
    print(f"\nSeed-variance summary across {len(metrics)} seeds:")
    display(summary.round(4))

    # Bar charts: mean ± std for each metric, with CV% annotated.
    cols = [c for c in metrics.columns if metrics[c].notna().any()]
    ncol = 4
    nrow = int(np.ceil(len(cols) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.2 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, cols):
        vals = metrics[c].dropna()
        ax.bar(vals.index.astype(str), vals.values, color="steelblue", alpha=0.85)
        m, s = vals.mean(), (vals.std(ddof=1) if len(vals) > 1 else 0.0)
        ax.axhline(m, color="k", ls="--", lw=1)
        ax.axhspan(m - s, m + s, color="orange", alpha=0.2)
        cv = 100 * s / abs(m) if m else float("nan")
        ax.set_title(f"{c}\nμ={m:.4g}  CV={cv:.2f}%", fontsize=10)
        ax.tick_params(axis="x", labelsize=8)
        ax.set_xlabel("seed", fontsize=8)
    for ax in axes[len(cols):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No eval metrics found yet — run the eval step of repro_experiment.sh.")

## How to read this

- **Part A** is a *correctness* check on our seeding. With `--deterministic`, same-seed runs
  should be bit-identical (weights) and identical (evals). If they are, any spread you see in
  Parts B/C is **purely** the effect of the seed, not kernel noise.
- **Part B** quantifies how stable the learned *dictionary* is across seeds. High cross-seed
  mean-max cosine (close to the same-seed reference) means features are reproducible up to
  permutation; a large gap below the red same-seed line means real seed-driven divergence.
- **Part C** gives the per-metric seed noise floor (CV%). When you later claim "method X beats
  method Y by Δ", Δ should comfortably exceed this seed CV to be credible.

**Reproduce:** `cd experiments && GPU=0 ./repro_experiment.sh` (override `NSEEDS`, `WIDTH`, `K`,
`TRAIN_TOKENS`, `DETERMINISTIC` as needed), then re-run this notebook top-to-bottom.